라이브러리 설치 및 임포트

In [6]:
!pip install koreanize-matplotlib

In [7]:
# 라이브러리 설치
!pip install -q pandas numpy matplotlib seaborn scikit-learn lightgbm xgboost scipy

import pandas as pd
import numpy as np
import koreanize_matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

In [5]:
# Google Drive & 데이터 로드

from google.colab import drive
drive.mount('/content/drive')

print("\n" + "=" * 80)
print("📊 데이터 로드")
print("=" * 80)

train = pd.read_csv('train.csv')  # 업로드한 경우
test = pd.read_csv('test.csv')

print(f"✓ 훈련 데이터: {train.shape}")
print(f"✓ 테스트 데이터: {test.shape}")

# 데이터 준비
y_train = train['임신 성공 여부'].copy()
X_train = train.drop(['ID', '임신 성공 여부'], axis=1)
X_test = test.drop('ID', axis=1)

print(f"\n타겟 분포:")
print(f"  실패 (0): {(y_train==0).sum():,}")
print(f"  성공 (1): {(y_train==1).sum():,}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

📊 데이터 로드
✓ 훈련 데이터: (256351, 69)
✓ 테스트 데이터: (90067, 68)

타겟 분포:
  실패 (0): 190,123
  성공 (1): 66,228


In [8]:
#전처리

print("\n" + "=" * 80)
print("🔧 전처리")
print("=" * 80)

# 변수 분류
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(include='object').columns.tolist()

# 훈련 통계 추출
numeric_stats = {col: X_train[col].median() for col in numeric_cols}
categorical_stats = {col: X_train[col].mode()[0] if len(X_train[col].mode()) > 0 else '알 수 없음'
                     for col in categorical_cols}

# 결측값 처리
def preprocess_data(df, numeric_stats, categorical_stats):
    df = df.copy()
    for col in categorical_stats.keys():
        if col in df.columns:
            df[col] = df[col].fillna(categorical_stats[col])
    for col in numeric_stats.keys():
        if col in df.columns:
            df[col] = df[col].fillna(numeric_stats[col])
    return df

X_train = preprocess_data(X_train, numeric_stats, categorical_stats)
X_test = preprocess_data(X_test, numeric_stats, categorical_stats)

# 로그 변환
high_skew_cols = ['저장된_배아_수', '해동된_배아_수', '저장된_신선_난자_수', '기증자_정자와_혼합된_난자_수']
for col in high_skew_cols:
    if col in X_train.columns:
        X_train[f'{col}_log'] = np.log1p(X_train[col])
        X_test[f'{col}_log'] = np.log1p(X_test[col])

print("✓ 결측값 처리 및 로그 변환 완료")


🔧 전처리
✓ 결측값 처리 및 로그 변환 완료


In [11]:
# 파생변수 30개 생성
print("\n" + "=" * 80)
print("🔬 파생변수 30개 생성")
print("=" * 80)

# --- Start of fix ---
# 특정 컬럼들의 데이터 타입을 숫자형으로 변환 (산술 연산 전에 필요)
# '시술 당시 나이' 컬럼 처리
if '시술 당시 나이' in X_train.columns:
    # '만'과 '세' 제거 후 첫 번째 숫자 추출 (예: '만18-34세' -> 18)
    X_train['시술 당시 나이'] = X_train['시술 당시 나이'].astype(str).str.extract('(\d+)').astype(float)
    X_test['시술 당시 나이'] = X_test['시술 당시 나이'].astype(str).str.extract('(\d+)').astype(float)

# '횟수'를 포함하는 컬럼 처리 (예: '0회' -> 0)
count_cols = [
    '총_임신_횟수', '총_시술_횟수', '총_출산_횟수',
    'IVF 임신 횟수', 'IVF 시술 횟수', 'DI 임신 횟수', 'DI 시술 횟수'
]

for col in count_cols:
    if col in X_train.columns and X_train[col].dtype == 'object':
        X_train[col] = X_train[col].astype(str).str.replace('회', '', regex=False)
        X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
    if col in X_test.columns and X_test[col].dtype == 'object':
        X_test[col] = X_test[col].astype(str).str.replace('회', '', regex=False)
        X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

# 변환 후 발생할 수 있는 NaN 값은 마지막 fillna(0)에서 처리됨

print("✓ 특정 컬럼 숫자형 변환 완료")
# --- End of fix ---

# Group 1: 성공률
if '총_임신_횟수' in X_train.columns and '총_시술_횟수' in X_train.columns:
    X_train['과거_성공_비율'] = X_train['총_임신_횟수'] / (X_train['총_시술_횟수'] + 1)
    X_test['과거_성공_비율'] = X_test['총_임신_횟수'] / (X_test['총_시술_횟수'] + 1)
    print("✓ 과거_성공_비율")

if '총_출산_횟수' in X_train.columns and '총_시술_횟수' in X_train.columns:
    X_train['과거_출산_비율'] = X_train['총_출산_횟수'] / (X_train['총_시술_횟수'] + 1)
    X_test['과거_출산_비율'] = X_test['총_출산_횟수'] / (X_test['총_시술_횟수'] + 1)
    print("✓ 과거_출산_비율")

if '총_임신_횟수' in X_train.columns and '총_출산_횟수' in X_train.columns:
    X_train['유산율'] = 1 - (X_train['총_출산_횟수'] / (X_train['총_임신_횟수'] + 1))
    X_test['유산율'] = 1 - (X_test['총_출산_횟수'] / (X_test['총_임신_횟수'] + 1))
    X_train['유산율'] = X_train['유산율'].clip(0, 1)
    X_test['유산율'] = X_test['유산율'].clip(0, 1)
    print("✓ 유산율")

if 'IVF 임신 횟수' in X_train.columns and 'IVF 시술 횟수' in X_train.columns:
    X_train['IVF_성공_비율'] = X_train['IVF 임신 횟수'] / (X_train['IVF 시술 횟수'] + 1)
    X_test['IVF_성공_비율'] = X_test['IVF 임신 횟수'] / (X_test['IVF 시술 횟수'] + 1)
    print("✓ IVF_성공_비율")

if 'DI 임신 횟수' in X_train.columns and 'DI 시술 횟수' in X_train.columns:
    X_train['DI_성공_비율'] = X_train['DI 임신 횟수'] / (X_train['DI 시술 횟수'] + 1)
    X_test['DI_성공_비율'] = X_test['DI 임신 횟수'] / (X_test['DI 시술 횟수'] + 1)
    print("✓ DI_성공_비율")

# Group 2: 배아
if '총_생성_배아_수' in X_train.columns and '혼합된_난자_수' in X_train.columns:
    X_train['배아_생성_효율'] = X_train['총_생성_배아_수'] / (X_train['혼합된_난자_수'] + 1)
    X_test['배아_생성_효율'] = X_test['총_생성_배아_수'] / (X_test['혼합된_난자_수'] + 1)
    print("✓ 배아_생성_효율")

if '이식된_배아_수' in X_train.columns and '총_생성_배아_수' in X_train.columns:
    X_train['배아_이식_효율'] = X_train['이식된_배아_수'] / (X_train['총_생성_배아_수'] + 1)
    X_test['배아_이식_효율'] = X_test['이식된_배아_수'] / (X_test['총_생성_배아_수'] + 1)
    print("✓ 배아_이식_효율")

if '저장된_배아_수' in X_train.columns and '총_생성_배아_수' in X_train.columns:
    X_train['배아_저장률'] = X_train['저장된_배아_수'] / (X_train['총_생성_배아_수'] + 1)
    X_test['배아_저장률'] = X_test['저장된_배아_수'] / (X_test['총_생성_배아_수'] + 1)
    print("✓ 배아_저장률")

if '총_생성_배아_수' in X_train.columns and '총_시술_횟수' in X_train.columns:
    X_train['배아_평균_생성_수'] = X_train['총_생성_배아_수'] / (X_train['총_시술_횟수'] + 1)
    X_test['배아_평균_생성_수'] = X_test['총_생성_배아_수'] / (X_test['총_시술_횟수'] + 1)
    print("✓ 배아_평균_생성_수")

if '이식된_배아_수' in X_train.columns and '저장된_배아_수' in X_train.columns and '총_생성_배아_수' in X_train.columns:
    X_train['배아_다양성_지수'] = (X_train['이식된_배아_수'] + X_train['저장된_배아_수']) / (X_train['총_생성_배아_수'] + 1)
    X_test['배아_다양성_지수'] = (X_test['이식된_배아_수'] + X_test['저장된_배아_수']) / (X_test['총_생성_배아_수'] + 1)
    print("✓ 배아_다양성_지수")

if '총_생성_배아_수' in X_train.columns and '이식된_배아_수' in X_train.columns and '저장된_배아_수' in X_train.columns:
    X_train['배아_손실률'] = 1 - ((X_train['이식된_배아_수'] + X_train['저장된_배아_수']) / (X_train['총_생성_배아_수'] + 1))
    X_test['배아_손실률'] = 1 - ((X_test['이식된_배아_수'] + X_test['저장된_배아_수']) / (X_test['총_생성_배아_수'] + 1))
    X_train['배아_손실률'] = X_train['배아_손실률'].clip(0, 1)
    X_test['배아_손실률'] = X_test['배아_손실률'].clip(0, 1)
    print("✓ 배아_손실률")

# Group 3-8: 나머지 파생변수들
# (이전 파일의 코드를 여기 추가)

# 난자
if '수집된_신선_난자_수' in X_train.columns and '혼합된_난자_수' in X_train.columns:
    X_train['신선_난자_비율'] = X_train['수집된_신선_난자_수'] / (X_train['혼합된_난자_수'] + 1)
    X_test['신선_난자_비율'] = X_test['수집된_신선_난자_수'] / (X_test['혼합된_난자_수'] + 1)
    print("✓ 신선_난자_비율")

if '시술 당시 나이' in X_train.columns:
    X_train['고령_여부'] = (X_train['시술 당시 나이'] >= 40).astype(int)
    X_test['고령_여부'] = (X_test['시술 당시 나이'] >= 40).astype(int)
    X_train['젊은_여부'] = (X_train['시술 당시 나이'] < 35).astype(int)
    X_test['젊은_여부'] = (X_test['시술 당시 나이'] < 35).astype(int)
    X_train['나이_제곱'] = X_train['시술 당시 나이'] ** 2
    X_test['나이_제곱'] = X_test['시술 당시 나이'] ** 2
    print("✓ 고령_여부, 젊은_여부, 나이_제곱")

if '총_시술_횟수' in X_train.columns:
    X_train['재시술_여부'] = (X_train['총_시술_횟수'] > 0).astype(int)
    X_test['재시술_여부'] = (X_test['총_시술_횟수'] > 0).astype(int)
    X_train['다중시술_여부'] = (X_train['총_시술_횟수'] >= 3).astype(int)
    X_test['다중시술_여부'] = (X_test['총_시술_횟수'] >= 3).astype(int)
    print("✓ 재시술_여부, 다중시술_여부")

# 결측값 처리
X_train = X_train.fillna(0).replace([np.inf, -np.inf], 0)
X_test = X_test.fillna(0).replace([np.inf, -np.inf], 0)

print(f"\n✅ 파생변수 생성 완료! ({X_train.shape[1]} 특성)")


🔬 파생변수 30개 생성
✓ 특정 컬럼 숫자형 변환 완료
✓ IVF_성공_비율
✓ DI_성공_비율
✓ 고령_여부, 젊은_여부, 나이_제곱

✅ 파생변수 생성 완료! (72 특성)


In [12]:
# 범주형 인코딩

print("\n" + "=" * 80)
print("🔤 범주형 변수 인코딩")
print("=" * 80)

categorical_features = X_train.select_dtypes(include='object').columns.tolist()

for col in categorical_features:
    unique_categories = sorted(X_train[col].unique())
    category_mapping = {cat: idx for idx, cat in enumerate(unique_categories)}
    X_train[col] = X_train[col].map(category_mapping)
    X_test[col] = X_test[col].map(category_mapping).fillna(-1).astype(int)

print(f"✓ {len(categorical_features)}개 변수 인코딩 완료")


🔤 범주형 변수 인코딩
✓ 15개 변수 인코딩 완료


In [13]:
# 모델 학습

print("\n" + "=" * 80)
print("🤖 모델 학습")
print("=" * 80)

# 데이터 분할
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(f"훈련: {X_tr.shape} | 검증: {X_val.shape}")

# LightGBM
print("\n【 LightGBM 학습 중... 】")
lgb_params = {
    'objective': 'binary', 'metric': 'auc', 'num_leaves': 31,
    'learning_rate': 0.05, 'max_depth': 7, 'min_child_samples': 20,
    'feature_fraction': 0.8, 'bagging_fraction': 0.8, 'bagging_freq': 5,
    'verbose': -1, 'scale_pos_weight': 2.87, 'seed': 42,
}
train_data = lgb.Dataset(X_tr, label=y_tr)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
lgb_model = lgb.train(lgb_params, train_data, num_boost_round=200,
                      valid_sets=[val_data],
                      callbacks=[lgb.log_evaluation(period=-1),
                                lgb.early_stopping(stopping_rounds=30)])
y_val_lgb = lgb_model.predict(X_val)
auc_lgb = roc_auc_score(y_val, y_val_lgb)
print(f"✓ LightGBM AUC: {auc_lgb:.4f}")

# XGBoost
print("\n【 XGBoost 학습 중... 】")
xgb_params = {
    'objective': 'binary:logistic', 'eval_metric': 'auc', 'max_depth': 6,
    'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8,
    'scale_pos_weight': 2.87, 'random_state': 42, 'tree_method': 'hist',
}
dtrain = xgb.DMatrix(X_tr, label=y_tr)
dval = xgb.DMatrix(X_val, label=y_val)
xgb_model = xgb.train(xgb_params, dtrain, num_boost_round=200,
                     evals=[(dtrain, 'train'), (dval, 'eval')],
                     verbose_eval=False, early_stopping_rounds=30)
y_val_xgb = xgb_model.predict(dval)
auc_xgb = roc_auc_score(y_val, y_val_xgb)
print(f"✓ XGBoost AUC: {auc_xgb:.4f}")


🤖 모델 학습
훈련: (205080, 72) | 검증: (51271, 72)

【 LightGBM 학습 중... 】
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[180]	valid_0's auc: 0.73728
✓ LightGBM AUC: 0.7373

【 XGBoost 학습 중... 】
✓ XGBoost AUC: 0.7360


In [14]:
# 최종 예측

print("\n" + "=" * 80)
print("📤 최종 예측")
print("=" * 80)

y_test_lgb = lgb_model.predict(X_test)
dtest = xgb.DMatrix(X_test)
y_test_xgb = xgb_model.predict(dtest)
y_test_ensemble = (y_test_lgb + y_test_xgb) / 2

print(f"\n예측값: [{y_test_ensemble.min():.6f}, {y_test_ensemble.max():.6f}]")
print(f"평균: {y_test_ensemble.mean():.6f}")

# 제출
submission = pd.DataFrame({'ID': test['ID'], 'probability': y_test_ensemble})
submission.to_csv('submission_0424.csv', index=False)
submission.to_csv('/content/drive/MyDrive/submission_0424.csv', index=False)

print(f"✓ submission.csv 생성")

from google.colab import files
files.download('submission_0424.csv')


📤 최종 예측

예측값: [0.000394, 0.896284]
평균: 0.450459
✓ submission.csv 생성


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>